# Load Dataset from HF

This will gather the dataset created using the dataset and load it into memory.


In [ ]:
# lib import
import os
from datasets import load_dataset, get_dataset_config_names

# setup
data = {}
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "wikipos")
configs = get_dataset_config_names("whatphiliptrains/wikipos")

for config in configs:
    data[config] = load_dataset(
        "whatphiliptrains/wikipos", config, cache_dir=source_ds_cache_dir
    )

test = data[configs[-1]]

# verification (optional)
print(f"Dataset length: {len(test['train'])}")
print(test["train"][0])
print(test["train"][-1])

Dataset length: 10000
{'id': '20104245', 'url': 'https://en.wikipedia.org/wiki/Processing%20delay', 'title': 'Processing delay', 'text': "In a network based on packet switching, processing delay is the time it takes routers to process the packet header. Processing delay is a key component in network delay.\n\nDuring processing of a packet, routers may check for bit-level errors in the packet that occurred during transmission as well as determining where the packet's next destination is. Processing delays in high-speed routers are typically on the order of microseconds or less. After this nodal processing, the router directs the packet to the queue where further delay can happen (queuing delay).\n\nIn the past, the processing delay has been ignored as insignificant compared to the other forms of network delay. However, in some systems, the processing delay can be quite large especially where routers are performing complex encryption algorithms and examining or modifying packet content. 

# Trustworthiness and Continuity

Using `scikit-learn` the trustworthiness (and continuity) between the dense embedding and the reduced position vector is calculated


In [2]:
import numpy as np
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors

n = 15
results = {}

def calculate_continuity(X, X_embedded, n_neighbors=5):
    """
    Calculate continuity metric for dimensionality reduction.
    
    Continuity measures whether points that are close in the low-dimensional 
    embedding are also close in the high-dimensional space.
    """
    # Ensure X is a numpy array
    if not isinstance(X, np.ndarray):
        X = np.array(X)
    if not isinstance(X_embedded, np.ndarray):
        X_embedded = np.array(X_embedded)
    
    # Find nearest neighbors in low-dimensional space
    nbrs_embedded = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X_embedded)
    _, indices_embedded = nbrs_embedded.kneighbors(X_embedded)
    
    # Find nearest neighbors in high-dimensional space
    nbrs_original = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X)
    _, indices_original = nbrs_original.kneighbors(X)
    
    continuity_sum = 0
    n_samples = X.shape[0]
    
    for i in range(n_samples):
        # Get k nearest neighbors in embedded space (excluding self)
        embedded_neighbors = set(indices_embedded[i][1:n_neighbors + 1])
        
        # Get k nearest neighbors in original space (excluding self)
        original_neighbors = set(indices_original[i][1:n_neighbors + 1])
        
        # Count how many embedded neighbors are also original neighbors
        intersection = len(embedded_neighbors.intersection(original_neighbors))
        continuity_sum += intersection / n_neighbors
    
    return continuity_sum / n_samples

for config in configs:
    current = data[config]["train"]
    pos = np.column_stack([current["x"], current["y"]])
    embeddings = np.array(current["embeddings"])

    trust = trustworthiness(X=embeddings, X_embedded=pos, n_neighbors=n)
    continuity = calculate_continuity(X=embeddings, X_embedded=pos, n_neighbors=n)

    results[config] = {"trustworthiness": trust, "continuity": continuity}
    print(f"[{config}] : trust: {trust:.4f}, continuity: {continuity:.4f}")

print(f"\nResults stored in 'results' dictionary with {len(results)} configurations.")

ImportError: cannot import name 'continuity' from 'sklearn.manifold' (c:\Users\ctechadmin\Documents\Philip\acme-paper\src\.venv\Lib\site-packages\sklearn\manifold\__init__.py)